##### Here are for most data analysis operations

In [11]:
import pandas as pd
from scipy import stats

In [3]:
# model/set/run list configurations
MODEL_GROUP = {
    "gpt-4.1": "gpt-4.1",
    "gpt-4.1-mini": "gpt-4.1",
    "gpt-4.1-nano": "gpt-4.1",
    "gpt-4o-mini": "gpt-4o",
    "gpt-4o-2024-11-20": "gpt-4o",
    "deepseek-chat-v3-0324": "deepseek-v3",
    "deepseek-r1-0528": "deepseek-r1",
    "gemini-2.5-flash": "gemini-2.5",
    "gemini-2.5-pro": "gemini-2.5",
    "llama-3.3-70b-instruct": "llama-3",
    "llama-3.1-8b-instruct": "llama-3",
    "llama-3.2-3b-instruct": "llama-3",
    "llama-3.2-1b-instruct": "llama-3",
    "llama-4-maverick": "llama-4",
    "llama-4-scout": "llama-4",
    "qwen3-30b-a3b": "qwen3",
    "qwen3-32b": "qwen3",
    "qwen3-8b": "qwen3",
    "qwen3-1.7b": "qwen3",
    "ernie-4.5-turbo-128k": "ernie-4.5",
    "ernie-4.5-0.3b": "ernie-4.5",
    "ernie-4.5-21b-a3b": "ernie-4.5"
}

In [4]:
df = pd.read_csv("lab/code_check.csv")

# every_test: dataframe with pass@k probability of every single model-set-run setup
every_test = (
    df.groupby(['model', 'set', 'lab'])
    .agg(pass_at_1=('result', lambda x: (x == "Pass").sum() / 164))
    .reset_index()
    .rename(columns={'pass_at_1': 'pass@1'})
)

every_test['series'] = every_test['model'].map(MODEL_GROUP)

every_test.to_csv("lab/pass@1.csv")
every_test

,model,set,lab,pass@1,series
0,deepseek-chat-v3-0324,interfered,lab01,0.902439,deepseek-v3
1,deepseek-chat-v3-0324,interfered,lab02,0.878049,deepseek-v3
2,deepseek-chat-v3-0324,interfered,lab03,0.902439,deepseek-v3
3,deepseek-chat-v3-0324,renamed,lab01,0.884146,deepseek-v3
4,deepseek-chat-v3-0324,renamed,lab02,0.896341,deepseek-v3
...,...,...,...,...,...
193,qwen3-8b,renamed,lab02,0.780488,qwen3
194,qwen3-8b,renamed,lab03,0.786585,qwen3
195,qwen3-8b,test,lab01,0.780488,qwen3
196,qwen3-8b,test,lab02,0.774390,qwen3


In [14]:
pass_at_1 = pd.read_csv("lab/pass@1.csv")

pivoted = (
    pass_at_1.pivot(
        index=['series', 'model', 'lab'],
        columns='set',
        values='pass@1'
    )
    .reset_index()
    .rename(
        columns={
            'test': 'pass@1_test',
            'renamed': 'pass@1_renamed',
            'interfered': 'pass@1_interfered'
        }
    )
)

pivoted.to_csv("lab/pass@1_pivoted.csv")
pivoted

set,series,model,lab,pass@1_interfered,pass@1_renamed,pass@1_test
0,deepseek-r1,deepseek-r1-0528,lab01,0.786585,0.676829,0.865854
1,deepseek-r1,deepseek-r1-0528,lab02,0.957317,0.823171,0.670732
2,deepseek-r1,deepseek-r1-0528,lab03,0.957317,0.835366,0.884146
3,deepseek-v3,deepseek-chat-v3-0324,lab01,0.902439,0.884146,0.932927
4,deepseek-v3,deepseek-chat-v3-0324,lab02,0.878049,0.896341,0.914634
...,...,...,...,...,...,...
61,qwen3,qwen3-32b,lab02,0.847561,0.823171,0.871951
62,qwen3,qwen3-32b,lab03,0.847561,0.798780,0.865854
63,qwen3,qwen3-8b,lab01,0.774390,0.817073,0.780488
64,qwen3,qwen3-8b,lab02,0.792683,0.780488,0.774390


In [49]:
pivoted.describe()

set,pass@1_interfered,pass@1_renamed,pass@1_test
count,66.000000,66.000000,66.000000
mean,0.704638,0.678492,0.722191
std,0.258605,0.253000,0.238539
min,0.000000,0.109756,0.201220
25%,0.522866,0.521341,0.562500
50%,0.847561,0.801829,0.847561
75%,0.902439,0.881098,0.902439
max,0.963415,0.945122,0.969512


In [29]:
W_test, p_shapiro_test = stats.shapiro(pivoted["pass@1_test"])
print(f"Shapiro–Wilk@test: W={W_test:.4f}, p={p_shapiro_test:.4f}")

W_renamed, p_shapiro_renamed = stats.shapiro(pivoted["pass@1_renamed"])
print(f"Shapiro–Wilk@renamed: W={W_renamed:.4f}, p={p_shapiro_renamed:.4f}")

W_interfered, p_shapiro_interfered = stats.shapiro(pivoted["pass@1_interfered"])
print(f"Shapiro–Wilk@interfered: W={W_interfered:.4f}, p={p_shapiro_interfered:.4f}")

Shapiro–Wilk@test: W=0.8251, p=0.0000
Shapiro–Wilk@renamed: W=0.8406, p=0.0000
Shapiro–Wilk@interfered: W=0.8357, p=0.0000


In [34]:
D_test, p_ks_test = stats.kstest(pivoted["pass@1_test"], "norm")
print(f"Kolmogorov-Smirnov@test: D={D_test:.4f}, p={p_ks_test:.4f}")

D_renamed, p_ks_renamed = stats.kstest(pivoted["pass@1_renamed"], "norm")
print(f"Kolmogorov-Smirnov@renamed: D={D_renamed:.4f}, p={p_ks_renamed:.4f}")

D_interfered, p_ks_interfered = stats.kstest(pivoted["pass@1_interfered"], "norm")
print(f"Kolmogorov-Smirnov@interfered: D={D_interfered:.4f}, p={p_ks_interfered:.4f}")

Kolmogorov-Smirnov@test: D=0.5797, p=0.0000
Kolmogorov-Smirnov@renamed: D=0.5437, p=0.0000
Kolmogorov-Smirnov@interfered: D=0.5660, p=0.0000


In [35]:
import statsmodels.api as sm

D_lilliefors_test, p_lilliefors_test = sm.stats.lilliefors(pivoted["pass@1_test"], "norm")
print(f"Lilliefors@test: D={D_lilliefors_test:.4f}, p={p_lilliefors_test:.4f}")

D_lilliefors_renamed, p_lilliefors_renamed = sm.stats.lilliefors(pivoted["pass@1_renamed"], "norm")
print(f"Lilliefors@renamed: D={D_lilliefors_renamed:.4f}, p={p_lilliefors_renamed:.4f}")

D_lilliefors_interfered, p_lilliefors_interfered = sm.stats.lilliefors(pivoted["pass@1_interfered"], "norm")
print(f"Lilliefors@interfered: D={D_lilliefors_interfered:.4f}, p={p_lilliefors_interfered:.4f}")

Lilliefors@test: D=0.2218, p=0.0010
Lilliefors@renamed: D=0.2172, p=0.0010
Lilliefors@interfered: D=0.2249, p=0.0010


In [36]:
import statsmodels.api as sm

D_lilliefors_test, p_lilliefors_test = sm.stats.lilliefors(pivoted["pass@1_test"], "norm")
print(f"Lilliefors@test: D={D_lilliefors_test:.4f}, p={p_lilliefors_test:.4f}")

D_lilliefors_renamed, p_lilliefors_renamed = sm.stats.lilliefors(pivoted["pass@1_renamed"], "norm")
print(f"Lilliefors@renamed: D={D_lilliefors_renamed:.4f}, p={p_lilliefors_renamed:.4f}")

D_lilliefors_interfered, p_lilliefors_interfered = sm.stats.lilliefors(pivoted["pass@1_interfered"], "norm")
print(f"Lilliefors@interfered: D={D_lilliefors_interfered:.4f}, p={p_lilliefors_interfered:.4f}")

Lilliefors@test: D=0.2218, p=0.0010
Lilliefors@renamed: D=0.2172, p=0.0010
Lilliefors@interfered: D=0.2249, p=0.0010


In [46]:
res_anderson_test = stats.anderson(pivoted["pass@1_test"], "norm")
print(f"Anderson@test: A²={res_anderson_test.statistic:.4f}, {res_anderson_test.significance_level[2]}% cv={res_anderson_test.critical_values[2]}")

res_anderson_renamed = stats.anderson(pivoted["pass@1_renamed"], "norm")
print(f"Anderson@renamed: A²={res_anderson_renamed.statistic:.4f}, {res_anderson_renamed.significance_level[2]}% cv={res_anderson_renamed.critical_values[2]}")

res_anderson_interfered = stats.anderson(pivoted["pass@1_interfered"], "norm")
print(f"Anderson@interfered: A²={res_anderson_interfered.statistic:.4f}, {res_anderson_interfered.significance_level[2]}% cv={res_anderson_interfered.critical_values[2]}")

Anderson@test: A²=4.5517, 5.0% cv=0.746
Anderson@renamed: A²=3.8956, 5.0% cv=0.746
Anderson@interfered: A²=4.3783, 5.0% cv=0.746


In [56]:
n = 66

W_wil_renamed, p_wil_renamed = stats.wilcoxon(pivoted['pass@1_test'], pivoted['pass@1_renamed'], alternative= "greater")
z_wil_renamed = (W_wil_renamed - (n * (n + 1) / 4)) / ((n * (n + 1) * (2 * n + 1) / 24) ** 0.5)
r_wil_renamed = abs(z_wil_renamed) / (n ** 0.5)
print(f"Wilcoxon signed-rank test@HumanEval>renamed: W={W_wil_renamed}, z={z_wil_renamed:.4f}, p={p_wil_renamed:.4f}, r={r_wil_renamed:.4f}")

W_wil_interfered, p_wil_interfered = stats.wilcoxon(pivoted['pass@1_test'], pivoted['pass@1_interfered'], alternative= "greater")
z_wil_interfered = (W_wil_interfered - (n * (n + 1) / 4)) / ((n * (n + 1) * (2 * n + 1) / 24) ** 0.5)
r_wil_interfered = abs(z_wil_interfered) / (n ** 0.5)
print(f"Wilcoxon signed-rank test@HumanEval>interfered: W={W_wil_interfered}, z={z_wil_interfered:.4f}, p={p_wil_interfered:.4f}, r={r_wil_interfered:.4f}")

Wilcoxon signed-rank test@HumanEval>renamed: W=1865.0, z=4.8517, p=0.0000, r=0.5972
Wilcoxon signed-rank test@HumanEval>interfered: W=1391.5, z=1.8270, p=0.0093, r=0.2249


In [12]:
import pandas as pd
from statsmodels.stats.contingency_tables import mcnemar

code_check = pd.read_csv("lab/code_check.csv")

df_test = code_check[code_check['set'] == 'test'].reset_index(drop=True)
df_renamed = code_check[code_check['set'] == 'renamed'].reset_index(drop=True)
df_interfered = code_check[code_check['set'] == 'interfered'].reset_index(drop=True)

labs = ["lab01", "lab02", "lab03"]
for lab in labs:
    print(f"Lab: {lab}")
    lab_test = df_test[df_test["lab"] == lab].reset_index(drop=True)
    lab_renamed = df_renamed[df_renamed["lab"] == lab].reset_index(drop=True)
    lab_interfered = df_interfered[df_interfered["lab"] == lab].reset_index(drop=True)

    cross_table_renamed = pd.crosstab(lab_test["result"] == "Pass", lab_renamed["result"] == "Pass",colnames=['Interfered Result'])
    mcnemar_renamed = mcnemar(cross_table_renamed, exact=False)
    print(cross_table_renamed)
    print(f"McNemar's test@{lab} renamed: statistic={mcnemar_renamed.statistic:.4f}, p-value={mcnemar_renamed.pvalue:.4f}")
    print("")
    mcnemar_interfered = mcnemar(cross_table_interfered, exact=False)
    cross_table_interfered = pd.crosstab(lab_test["result"] == "Pass", lab_interfered["result"] == "Pass", colnames=['Interfered Result'])
    print(cross_table_interfered)
    print(f"McNemar's test@{lab} interfered: statistic={mcnemar_interfered.statistic:.4f}, p-value={mcnemar_interfered.pvalue:.4f}")
    print("")

Lab: lab01
Interfered Result  False  True 
result                         
False                720    200
True                 446   2242
McNemar's test@lab01 renamed: statistic=92.9180, p-value=0.0000

Interfered Result  False  True 
result                         
False                698    222
True                 357   2331
McNemar's test@lab01 interfered: statistic=10.9113, p-value=0.0010

Lab: lab02
Interfered Result  False  True 
result                         
False                809    250
True                 340   2209
McNemar's test@lab02 renamed: statistic=13.4254, p-value=0.0002

Interfered Result  False  True 
result                         
False                804    255
True                 238   2311
McNemar's test@lab02 interfered: statistic=31.0121, p-value=0.0000

Lab: lab03
Interfered Result  False  True 
result                         
False                813    215
True                 352   2228
McNemar's test@lab03 renamed: statistic=32.6208, p-value=0.00